In [1]:
import pandas as pd
import numpy as np
import requests
import json
import time
import logging
from pathlib import Path
from datetime import datetime

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_colwidth", 60)

PROJECT_DIR = Path().resolve().parent
RAW_DIR     = PROJECT_DIR / "data" / "raw"
PROC_DIR    = PROJECT_DIR / "data" / "processed"
DB_PATH     = PROJECT_DIR / "data" / "db" / "bluestock.db"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

log.info(f"PROJECT_DIR : {PROJECT_DIR}")
log.info(f"RAW_DIR     : {RAW_DIR}")
log.info(f"PROC_DIR    : {PROC_DIR}")

2026-06-02 22:18:58 | INFO | PROJECT_DIR : D:\Engsci_Study_materials\INTERNSHIP\Capstone_Project
2026-06-02 22:18:58 | INFO | RAW_DIR     : D:\Engsci_Study_materials\INTERNSHIP\Capstone_Project\data\raw
2026-06-02 22:18:58 | INFO | PROC_DIR    : D:\Engsci_Study_materials\INTERNSHIP\Capstone_Project\data\processed


In [2]:
DATASETS = {
    "fund_master"          : "01_fund_master.csv",
    "nav_history"          : "02_nav_history.csv",
    "aum_by_fund_house"    : "03_aum_by_fund_house.csv",
    "monthly_sip_inflows"  : "04_monthly_sip_inflows.csv",
    "category_inflows"     : "05_category_inflows.csv",
    "industry_folio_count" : "06_industry_folio_count.csv",
    "scheme_performance"   : "07_scheme_performance.csv",
    "investor_transactions": "08_investor_transactions.csv",
    "portfolio_holdings"   : "09_portfolio_holdings.csv",
    "benchmark_indices"    : "10_benchmark_indices.csv",
}

DATE_COLUMNS = {
    "fund_master"          : ["launch_date"],
    "nav_history"          : ["date"],
    "aum_by_fund_house"    : ["date"],
    "monthly_sip_inflows"  : ["month"],
    "category_inflows"     : ["month"],
    "industry_folio_count" : ["month"],
    "investor_transactions": ["transaction_date"],
    "portfolio_holdings"   : ["portfolio_date"],
    "benchmark_indices"    : ["date"],
}

dfs          = {}
load_summary = []

log.info("[STEP 1] Loading datasets ...")

for name, fname in DATASETS.items():
    path = RAW_DIR / fname
    if not path.exists():
        log.warning(f"File not found: {path}")
        load_summary.append({"dataset": name, "rows": 0, "cols": 0, "status": "NOT FOUND"})
        continue
    try:
        df = pd.read_csv(path, low_memory=False)
        for col in DATE_COLUMNS.get(name, []):
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors="coerce")
        dfs[name] = df
        load_summary.append({"dataset": name, "rows": df.shape[0], "cols": df.shape[1], "status": "OK"})
        log.info(f"  Loaded '{name}' — {df.shape[0]:,} rows x {df.shape[1]} cols")
    except Exception as e:
        log.error(f"  Could not load '{name}': {e}")
        load_summary.append({"dataset": name, "rows": 0, "cols": 0, "status": f"ERROR: {e}"})

log.info(f"  {len(dfs)}/{len(DATASETS)} datasets loaded.")
pd.DataFrame(load_summary)

2026-06-02 22:19:08 | INFO | [STEP 1] Loading datasets ...
2026-06-02 22:19:08 | INFO |   Loaded 'fund_master' — 40 rows x 15 cols
2026-06-02 22:19:08 | INFO |   Loaded 'nav_history' — 46,000 rows x 3 cols
2026-06-02 22:19:08 | INFO |   Loaded 'aum_by_fund_house' — 90 rows x 5 cols
2026-06-02 22:19:08 | INFO |   Loaded 'monthly_sip_inflows' — 48 rows x 6 cols
2026-06-02 22:19:08 | INFO |   Loaded 'category_inflows' — 144 rows x 3 cols
2026-06-02 22:19:08 | INFO |   Loaded 'industry_folio_count' — 21 rows x 6 cols
2026-06-02 22:19:08 | INFO |   Loaded 'scheme_performance' — 40 rows x 19 cols
2026-06-02 22:19:09 | INFO |   Loaded 'investor_transactions' — 32,778 rows x 13 cols
2026-06-02 22:19:09 | INFO |   Loaded 'portfolio_holdings' — 322 rows x 8 cols
2026-06-02 22:19:09 | INFO |   Loaded 'benchmark_indices' — 8,050 rows x 3 cols
2026-06-02 22:19:09 | INFO |   10/10 datasets loaded.


,dataset,rows,cols,status
0,fund_master,40,15,OK
1,nav_history,46000,3,OK
2,aum_by_fund_house,90,5,OK
3,monthly_sip_inflows,48,6,OK
4,category_inflows,144,3,OK
5,industry_folio_count,21,6,OK
6,scheme_performance,40,19,OK
7,investor_transactions,32778,13,OK
8,portfolio_holdings,322,8,OK
9,benchmark_indices,8050,3,OK


In [5]:
log.info("[STEP 2] Profiling datasets ...")

for name, df in dfs.items():
    print(f"\n{'='*60}")
    print(f"  DATASET: {name.upper()}")
    print(f"{'='*60}")
    print(f"\nShape: {df.shape[0]:,} rows x {df.shape[1]} cols")

    print("\nData Types:")
    display(df.dtypes.to_frame("dtype"))

    print("\nFirst 3 Rows:")
    display(df.head(3))

    null_counts = df.isnull().sum()
    null_pct    = (null_counts / len(df) * 100).round(2)
    null_df     = pd.DataFrame({"null_count": null_counts, "null_%": null_pct})
    null_df     = null_df[null_df["null_count"] > 0]

    if not null_df.empty:
        print("\nNull Values:")
        display(null_df)
    else:
        print("\nNo null values.")

    dup = df.duplicated().sum()
    print(f"\nDuplicate Rows: {dup:,}")

    num_cols = df.select_dtypes(include="number").columns.tolist()
    if num_cols:
        print("\nNumeric Summary:")
        display(df[num_cols].describe().round(4))

    print(f"\n{'─'*60}")

2026-06-02 22:19:49 | INFO | [STEP 2] Profiling datasets ...



  DATASET: FUND_MASTER

Shape: 40 rows x 15 cols

Data Types:


,dtype
amfi_code,int64
fund_house,object
scheme_name,object
category,object
sub_category,object
plan,object
launch_date,datetime64[ns]
benchmark,object
expense_ratio_pct,float64
exit_load_pct,float64



First 3 Rows:


,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.5400,1.0000,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.6600,1.0000,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.4300,1.0000,500,1000,R. Srinivasan,Very High,EC03


...


First 3 Rows:


,date,index_name,close_value
0,2022-01-03,NIFTY50,"17,492.7900"
1,2022-01-04,NIFTY50,"17,689.6400"
2,2022-01-05,NIFTY50,"17,835.0500"



No null values.

Duplicate Rows: 0

Numeric Summary:


,close_value
count,"8,050.0000"
mean,"18,351.9502"
std,"14,169.9951"
min,"1,444.1300"
25%,"2,819.0625"
50%,"18,392.9600"
75%,"26,539.0975"
max,"79,075.3900"



────────────────────────────────────────────────────────────


In [6]:
log.info("[STEP 3] Exploring fund_master ...")

if "fund_master" in dfs:
    fm = dfs["fund_master"]

    for label, col in [
        ("Fund Houses",    "fund_house"),
        ("Categories",     "category"),
        ("Sub-Categories", "sub_category"),
        ("Risk Grades",    "risk_category"),
    ]:
        if col in fm.columns:
            vals = fm[col].dropna().unique()
            print(f"\n{label} ({len(vals)}):")
            display(
                fm[col].value_counts().rename_axis(col).reset_index(name="schemes")
            )

    codes = pd.to_numeric(fm["amfi_code"], errors="coerce").dropna()
    print(f"\nAMFI Code Range: {int(codes.min())} — {int(codes.max())}  |  Unique: {codes.nunique():,}")
    if len(codes) != codes.nunique():
        print(f"  WARNING: {len(codes) - codes.nunique():,} duplicate AMFI codes found!")
else:
    log.warning("  fund_master not loaded — skipping exploration.")

2026-06-02 22:19:54 | INFO | [STEP 3] Exploring fund_master ...



Fund Houses (10):


,fund_house,schemes
0,SBI Mutual Fund,5
1,HDFC Mutual Fund,5
2,ICICI Prudential MF,5
3,Nippon India MF,5
4,Kotak Mahindra MF,4
5,Axis Mutual Fund,4
6,Aditya Birla Sun Life MF,3
7,UTI Mutual Fund,3
8,Mirae Asset MF,3
9,DSP Mutual Fund,3



Categories (2):


,category,schemes
0,Equity,34
1,Debt,6



Sub-Categories (12):


,sub_category,schemes
0,Large Cap,14
1,Mid Cap,7
2,Small Cap,6
3,Liquid,3
4,Gilt,2
5,Flexi Cap,2
6,Short Duration,1
7,Value,1
8,Index/ETF,1
9,Index,1



Risk Grades (5):


,risk_category,schemes
0,Moderate,16
1,High,8
2,Very High,6
3,Low,6
4,Moderately High,4



AMFI Code Range: 100016 — 149324  |  Unique: 40


In [7]:
SCHEMES = {
    125497: "HDFC_Top100_Direct",
    119551: "SBI_Bluechip_Direct",
    120503: "ICICI_Bluechip_Direct",
    118632: "Nippon_LargeCap_Direct",
    119092: "Axis_Bluechip_Direct",
    120841: "Kotak_Bluechip_Direct",
}

BASE_URL = "https://api.mfapi.in/mf"
TIMEOUT  = 15
DELAY    = 0.5

live_dfs = []

log.info("[STEP 4] Fetching live NAV data from mfapi.in ...")

for code, name in SCHEMES.items():
    try:
        response = requests.get(f"{BASE_URL}/{code}", timeout=TIMEOUT)
        response.raise_for_status()
        payload = response.json()

        if payload.get("status") != "SUCCESS":
            log.warning(f"  API returned non-SUCCESS for {name}: {payload.get('status')}")
            continue

        meta        = payload.get("meta", {})
        nav_records = payload.get("data", [])

        if not nav_records:
            log.warning(f"  No NAV records returned for {name}.")
            continue

        df = pd.DataFrame(nav_records)
        df["amfi_code"]       = code
        df["scheme_name"]     = meta.get("scheme_name", name)
        df["fund_house"]      = meta.get("fund_house", "")
        df["scheme_type"]     = meta.get("scheme_type", "")
        df["scheme_category"] = meta.get("scheme_category", "")

        df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y", dayfirst=True)
        df["nav"]  = pd.to_numeric(df["nav"], errors="coerce")
        df = df.sort_values("date").reset_index(drop=True)
        df = df[df["nav"] > 0]

        full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
        df = (
            df.set_index("date")
            .reindex(full_range)
            .assign(
                amfi_code       = code,
                scheme_name     = df["scheme_name"].iloc[0],
                fund_house      = df["fund_house"].iloc[0],
                scheme_type     = df["scheme_type"].iloc[0],
                scheme_category = df["scheme_category"].iloc[0],
            )
        )
        df["nav"] = df["nav"].ffill()
        df = df.dropna(subset=["nav"]).reset_index().rename(columns={"index": "date"})

        out_path = RAW_DIR / f"nav_{name}.csv"
        df.to_csv(out_path, index=False)
        live_dfs.append(df)

        log.info(
            f"  {name}: {len(df):,} records | "
            f"{df['date'].min().date()} -> {df['date'].max().date()} | "
            f"Latest NAV: {df['nav'].iloc[-1]:.4f}"
        )

    except requests.exceptions.Timeout:
        log.error(f"  Timeout fetching {name} (code: {code})")
    except requests.exceptions.HTTPError as e:
        log.error(f"  HTTP error for {name}: {e}")
    except requests.exceptions.RequestException as e:
        log.error(f"  Network error for {name}: {e}")
    except (KeyError, ValueError, json.JSONDecodeError) as e:
        log.error(f"  Parse error for {name}: {e}")

    time.sleep(DELAY)

if live_dfs:
    combined      = pd.concat(live_dfs, ignore_index=True)
    combined_path = RAW_DIR / "nav_all_schemes_live.csv"
    combined.to_csv(combined_path, index=False)
    log.info(f"  Combined CSV saved -> {combined_path.relative_to(PROJECT_DIR)}")
    log.info(f"  Total rows: {len(combined):,}")
    display(
        combined.groupby("scheme_name").agg(
            records    =("nav",  "count"),
            start_date =("date", "min"),
            end_date   =("date", "max"),
            latest_nav =("nav",  "last"),
        ).round(4)
    )

2026-06-02 22:19:57 | INFO | [STEP 4] Fetching live NAV data from mfapi.in ...
2026-06-02 22:19:58 | INFO |   HDFC_Top100_Direct: 4,579 records | 2013-11-18 -> 2026-06-01 | Latest NAV: 192.3195
2026-06-02 22:19:58 | INFO |   SBI_Bluechip_Direct: 4,899 records | 2013-01-02 -> 2026-06-01 | Latest NAV: 104.7025
2026-06-02 22:19:59 | INFO |   ICICI_Bluechip_Direct: 4,899 records | 2013-01-02 -> 2026-06-01 | Latest NAV: 103.0948
2026-06-02 22:20:00 | INFO |   Nippon_LargeCap_Direct: 4,899 records | 2013-01-02 -> 2026-06-01 | Latest NAV: 97.1944
2026-06-02 22:20:01 | INFO |   Axis_Bluechip_Direct: 4,901 records | 2012-12-31 -> 2026-06-01 | Latest NAV: 6156.7532
2026-06-02 22:20:02 | INFO |   Kotak_Bluechip_Direct: 4,894 records | 2013-01-07 -> 2026-06-01 | Latest NAV: 246.9814
2026-06-02 22:20:03 | INFO |   Combined CSV saved -> data\raw\nav_all_schemes_live.csv
2026-06-02 22:20:03 | INFO |   Total rows: 29,071


,records,start_date,end_date,latest_nav
scheme_name,,,,
Aditya Birla Sun Life Banking & PSU Debt Fund - DIRECT - IDCW,4899,2013-01-02,2026-06-01,104.7025
Axis ELSS Tax Saver Fund - Direct Plan - Growth Option,4899,2013-01-02,2026-06-01,103.0948
HDFC Money Market Fund - Growth Option - Direct Plan,4901,2012-12-31,2026-06-01,"6,156.7532"
Nippon India Large Cap Fund - Direct Plan Growth Plan - Growth Option,4899,2013-01-02,2026-06-01,97.1944
SBI Small Cap Fund - Direct Plan - Growth,4579,2013-11-18,2026-06-01,192.3195
quant Mid Cap Fund - Growth Option - Direct Plan,4894,2013-01-07,2026-06-01,246.9814


In [4]:
log.info("[STEP 5] Validating AMFI codes ...")

if "fund_master" in dfs and "nav_history" in dfs:
    fm  = dfs["fund_master"]
    nav = dfs["nav_history"]

    fm_codes  = set(pd.to_numeric(fm["amfi_code"],  errors="coerce").dropna().astype(int))
    nav_codes = set(pd.to_numeric(nav["amfi_code"], errors="coerce").dropna().astype(int))

    matched  = fm_codes & nav_codes
    missing  = fm_codes - nav_codes
    extra    = nav_codes - fm_codes
    coverage = round(len(matched) / len(fm_codes) * 100, 2) if fm_codes else 0

    print(f"\n{'='*60}")
    print("  AMFI CODE VALIDATION")
    print(f"{'='*60}")

    validation_df = pd.DataFrame([
        {"Metric": "Fund Master codes",      "Count": len(fm_codes)},
        {"Metric": "NAV History codes",      "Count": len(nav_codes)},
        {"Metric": "Matched (both)",          "Count": len(matched)},
        {"Metric": "In FM, missing from NAV", "Count": len(missing)},
        {"Metric": "In NAV, not in FM",       "Count": len(extra)},
        {"Metric": "Coverage %",              "Count": coverage},
    ])
    display(validation_df)

    quality_flag = "PASS" if coverage >= 90 else ("WARN" if coverage >= 70 else "FAIL")
    print(f"\n  Coverage: {coverage}%  |  Quality: {quality_flag}")

    if missing:
        print(f"  FM codes missing from NAV (first 10): {sorted(missing)[:10]}")
else:
    missing_keys = [k for k in ("fund_master", "nav_history") if k not in dfs]
    log.warning(f"  Skipped — missing: {missing_keys}")

2026-06-02 22:19:30 | INFO | [STEP 5] Validating AMFI codes ...



  AMFI CODE VALIDATION


,Metric,Count
0,Fund Master codes,40.0000
1,NAV History codes,40.0000
2,Matched (both),40.0000
3,"In FM, missing from NAV",0.0000
4,"In NAV, not in FM",0.0000
5,Coverage %,100.0000



  Coverage: 100.0%  |  Quality: PASS


In [8]:
log.info("[STEP 6] Building data quality summary ...")

summary_rows = []
for name, df in dfs.items():
    null_cols   = int((df.isnull().sum() > 0).sum())
    total_nulls = int(df.isnull().sum().sum())
    dup_rows    = int(df.duplicated().sum())
    n_anomalies = sum([
        dup_rows > 0,
        total_nulls > 0,
    ])
    flag = "HIGH" if n_anomalies > 2 else ("MEDIUM" if n_anomalies else "LOW")
    summary_rows.append({
        "Dataset"    : name,
        "Rows"       : df.shape[0],
        "Columns"    : df.shape[1],
        "Null Cols"  : null_cols,
        "Total Nulls": total_nulls,
        "Dup Rows"   : dup_rows,
        "Quality"    : flag,
    })

qdf = pd.DataFrame(summary_rows)
display(qdf)

out_path = PROC_DIR / "data_quality_summary.csv"
qdf.to_csv(out_path, index=False)
log.info(f"  Saved -> {out_path.relative_to(PROJECT_DIR)}")

total_rows  = qdf["Rows"].sum()
total_nulls = qdf["Total Nulls"].sum()

log.info("\n" + "=" * 60)
log.info("  INGESTION COMPLETE")
log.info(f"  Datasets loaded   : {len(dfs)}")
log.info(f"  Total rows        : {total_rows:,}")
log.info(f"  Total null values : {total_nulls:,}")
log.info("=" * 60)

2026-06-02 22:20:08 | INFO | [STEP 6] Building data quality summary ...


,Dataset,Rows,Columns,Null Cols,Total Nulls,Dup Rows,Quality
0,fund_master,40,15,0,0,0,LOW
1,nav_history,46000,3,0,0,0,LOW
2,aum_by_fund_house,90,5,0,0,0,LOW
3,monthly_sip_inflows,48,6,1,12,0,MEDIUM
4,category_inflows,144,3,0,0,0,LOW
5,industry_folio_count,21,6,0,0,0,LOW
6,scheme_performance,40,19,0,0,0,LOW
7,investor_transactions,32778,13,0,0,0,LOW
8,portfolio_holdings,322,8,0,0,0,LOW
9,benchmark_indices,8050,3,0,0,0,LOW


2026-06-02 22:20:09 | INFO |   Saved -> data\processed\data_quality_summary.csv
2026-06-02 22:20:09 | INFO | 
2026-06-02 22:20:09 | INFO |   INGESTION COMPLETE
2026-06-02 22:20:09 | INFO |   Datasets loaded   : 10
2026-06-02 22:20:09 | INFO |   Total rows        : 87,533
2026-06-02 22:20:09 | INFO |   Total null values : 12
2026-06-02 22:20:09 | INFO | ============================================================
